<a href="https://colab.research.google.com/github/Sathwik828/renifrocment-Learinig-Lab/blob/main/Rl11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Practical 11: Policy Iteration in a Deterministic Grid World

# 1. Define states, actions, and parameters
states = ["S0", "S1", "G"]
actions = ["Up", "Down", "Left", "Right"]

gamma = 0.9
theta = 1e-6
goal_reward = 100

# Grid:
# S0 | WALL
# S1 | G

# 2. Define deterministic transitions and rewards
# Wall/boundary: remain in the same state, reward = -1
# Goal: reward = +100, terminal state value = 0

def get_transition(state, action):
    transitions = {
        "S0": {
            "Up":    ("S0", -1),
            "Down":  ("S1", -1),
            "Left":  ("S0", -1),
            "Right": ("S0", -1)
        },
        "S1": {
            "Up":    ("S0", -1),
            "Down":  ("S1", -1),
            "Left":  ("S1", -1),
            "Right": ("G", goal_reward)
        }
    }

    if state == "G":
        return "G", 0

    return transitions[state][action]


# 3. Inspect the MDP
print("STATES:", states)
print("ACTIONS:", actions)
print("Discount factor gamma:", gamma)
print("Normal movement reward: -1")
print("Wall/boundary reward: -1")
print("Goal reward:", goal_reward)
print("\nTRANSITION MAPPING")

for state in ["S0", "S1"]:
    for action in actions:
        next_state, reward = get_transition(state, action)
        print(
            f"{state} --{action}--> {next_state}, "
            f"Reward = {reward}"
        )

print("\nAll state-action transitions are defined.")


# 4. Initialize policy and state values
policy = {
    "S0": "Up",
    "S1": "Up"
}

V = {
    "S0": 0.0,
    "S1": 0.0,
    "G": 0.0
}

print("\nINITIAL POLICY:", policy)
print("INITIAL STATE VALUES:", V)


# 5. Policy Iteration
policy_iteration = 0

while True:
    policy_iteration += 1

    print(f"\n{'=' * 55}")
    print(f"POLICY ITERATION {policy_iteration}")
    print("Current policy:", policy)

    # POLICY EVALUATION
    # Evaluate the current policy until convergence.

    evaluation_iteration = 0

    while True:
        evaluation_iteration += 1
        old_V = V.copy()
        new_V = {"G": 0.0}

        # Synchronous updates: use only old_V
        for state in ["S0", "S1"]:
            action = policy[state]
            next_state, reward = get_transition(state, action)

            new_V[state] = (
                reward + gamma * old_V[next_state]
            )

        delta = max(
            abs(new_V[state] - old_V[state])
            for state in states
        )

        V = new_V

        print(
            f"Evaluation {evaluation_iteration}: "
            f"V(S0)={V['S0']:.6f}, "
            f"V(S1)={V['S1']:.6f}, "
            f"V(G)={V['G']:.6f}, "
            f"Delta={delta:.6f}"
        )

        if delta < theta:
            break

    # ACTION-VALUE CALCULATION
    Q = {}

    for state in ["S0", "S1"]:
        Q[state] = {}

        for action in actions:
            next_state, reward = get_transition(state, action)

            Q[state][action] = (
                reward + gamma * V[next_state]
            )

    print("\nACTION VALUES Q(s, a):")

    for state in ["S0", "S1"]:
        for action in actions:
            print(
                f"Q({state}, {action}) = "
                f"{Q[state][action]:.6f}"
            )

    # POLICY IMPROVEMENT
    old_policy = policy.copy()

    for state in ["S0", "S1"]:
        best_action = max(
            actions,
            key=lambda action: Q[state][action]
        )
        policy[state] = best_action

    print("\nImproved policy:", policy)

    # POLICY STABILITY CHECK
    if policy == old_policy:
        print("Policy is stable. Algorithm converged.")
        break


# 6. Display final optimal state values
print("\nFINAL OPTIMAL STATE VALUES")

for state in states:
    print(f"V*({state}) = {V[state]:.6f}")


# 7. Display final optimal action values
print("\nFINAL ACTION-VALUE TABLE")

for state in ["S0", "S1"]:
    for action in actions:
        next_state, reward = get_transition(state, action)
        q_value = reward + gamma * V[next_state]

        print(
            f"Q*({state}, {action}) = {q_value:.6f}"
        )


# 8. Display the optimal policy
print("\nOPTIMAL POLICY")

for state in ["S0", "S1"]:
    print(f"{state}: {policy[state]}")


# 9. Verify Bellman Optimality
print("\nBELLMAN OPTIMALITY VERIFICATION")

for state in ["S0", "S1"]:
    q_values = {}

    for action in actions:
        next_state, reward = get_transition(state, action)
        q_values[action] = reward + gamma * V[next_state]

    max_q = max(q_values.values())

    print(
        f"{state}: V* = {V[state]:.6f}, "
        f"max Q* = {max_q:.6f}"
    )

    assert abs(V[state] - max_q) < 1e-4

print("\nVerification successful!")
print("Optimal path: S0 -> S1 -> G")


STATES: ['S0', 'S1', 'G']
ACTIONS: ['Up', 'Down', 'Left', 'Right']
Discount factor gamma: 0.9
Normal movement reward: -1
Wall/boundary reward: -1
Goal reward: 100

TRANSITION MAPPING
S0 --Up--> S0, Reward = -1
S0 --Down--> S1, Reward = -1
S0 --Left--> S0, Reward = -1
S0 --Right--> S0, Reward = -1
S1 --Up--> S0, Reward = -1
S1 --Down--> S1, Reward = -1
S1 --Left--> S1, Reward = -1
S1 --Right--> G, Reward = 100

All state-action transitions are defined.

INITIAL POLICY: {'S0': 'Up', 'S1': 'Up'}
INITIAL STATE VALUES: {'S0': 0.0, 'S1': 0.0, 'G': 0.0}

POLICY ITERATION 1
Current policy: {'S0': 'Up', 'S1': 'Up'}
Evaluation 1: V(S0)=-1.000000, V(S1)=-1.000000, V(G)=0.000000, Delta=1.000000
Evaluation 2: V(S0)=-1.900000, V(S1)=-1.900000, V(G)=0.000000, Delta=0.900000
Evaluation 3: V(S0)=-2.710000, V(S1)=-2.710000, V(G)=0.000000, Delta=0.810000
Evaluation 4: V(S0)=-3.439000, V(S1)=-3.439000, V(G)=0.000000, Delta=0.729000
Evaluation 5: V(S0)=-4.095100, V(S1)=-4.095100, V(G)=0.000000, Delta=0.656